# Laboratório Prático 2 - Dashboard Analítico de Vendas Globais

## Importando as bibliotecas

In [1]:
#install
#!pip install -q -U duckdb matplotlib numpy pandas plotly scikit-learn scipy seaborn shap statsmodels xgboost 

In [2]:
#imports
import pandas as pd
import numpy as np
import os
import duckdb as ddb
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import plotly.graph_objects as go

## Acessando os dados

In [3]:
pasta_com_arquivos = '1-datasets/'
for nome_do_arquivo in os.listdir(pasta_com_arquivos):
    caminho_do_arquivo = os.path.join(pasta_com_arquivos, nome_do_arquivo)
    if nome_do_arquivo[-10:] == '_clear.csv':
        variavel_nome = nome_do_arquivo.replace('_clear.csv','').replace('-','_').replace(' ','_')
        try:
            globals()[f'dataset_{variavel_nome}'] = pd.read_csv(caminho_do_arquivo)
        except Exception as e:
            print(f'Falha no carregamento no arquivo {nome_do_arquivo}: {e}')

In [4]:
dataset_Clientes.head(1) 

,ID Cliente,Nome Cliente,Segmento,Cidade,Estado,Pais,Mercado,Regiao
0,RH-19495,Rick Hansen,Consumidor,New York City,New York,United States,US,East


In [5]:
dataset_Clientes.columns 

Index(['ID Cliente', 'Nome Cliente', 'Segmento', 'Cidade', 'Estado', 'Pais',
       'Mercado', 'Regiao'],
      dtype='str')

In [6]:
dataset_Pedidos.head(1) 

,ID Pedido,Data Pedido,Data Envio,Modo Envio,Prioridade Pedido
0,CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico


In [7]:
colunas_para_conversao = ['Data Envio', 'Data Pedido']
for coluna in colunas_para_conversao:
    dataset_Pedidos[coluna] = pd.to_datetime(dataset_Pedidos[coluna], errors= 'coerce', format='%d-%m-%Y')

In [8]:
dataset_Pedidos.columns 

Index(['ID Pedido', 'Data Pedido', 'Data Envio', 'Modo Envio',
       'Prioridade Pedido'],
      dtype='str')

In [9]:
dataset_Produtos.head(1)

,ID Produto,Categoria,SubCategoria,Nome Produto
0,TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...


In [10]:
dataset_Produtos.columns

Index(['ID Produto', 'Categoria', 'SubCategoria', 'Nome Produto'], dtype='str')

In [11]:
dataset_Vendas.head(1)

,Pedido,Cliente,Produto,Valor Venda,Quantidade Vendida,Custo Envio
0,CA-2012-124891,RH-19495,TEC-AC-10003033,2309.65,7,933.57


In [12]:
dataset_Vendas.columns

Index(['Pedido', 'Cliente', 'Produto', 'Valor Venda', 'Quantidade Vendida',
       'Custo Envio'],
      dtype='str')

## Conexão do DuckDB

In [13]:
conexao_db = ddb.connect()

In [14]:
conexao_db.register('Cliente', dataset_Clientes)
conexao_db.register('Pedidos',dataset_Pedidos)
conexao_db.register('Produtos',dataset_Produtos)
conexao_db.register('Vendas',dataset_Vendas)

### Super Dataset

In [15]:
super_dataset = ddb.query("""
    SELECT 
        c.*,
        p.*,
        pr.*,
        v.*
    FROM dataset_Vendas v
    JOIN dataset_Clientes c  ON v."Cliente" = c."ID Cliente"
    JOIN dataset_Pedidos p   ON v."Pedido"  = p."ID Pedido"
    JOIN dataset_Produtos pr ON v."Produto" = pr."ID Produto"
""").df()

In [16]:
super_dataset.columns

Index(['ID Cliente', 'Nome Cliente', 'Segmento', 'Cidade', 'Estado', 'Pais',
       'Mercado', 'Regiao', 'ID Pedido', 'Data Pedido', 'Data Envio',
       'Modo Envio', 'Prioridade Pedido', 'ID Produto', 'Categoria',
       'SubCategoria', 'Nome Produto', 'Pedido', 'Cliente', 'Produto',
       'Valor Venda', 'Quantidade Vendida', 'Custo Envio'],
      dtype='str')

In [17]:
colunas_organizadas = [
    # Cliente
    'ID Cliente', 'Nome Cliente', 'Segmento',
    'Cidade', 'Estado', 'Pais', 'Mercado', 'Regiao',
    
    # Pedido
    'ID Pedido', 'Data Pedido', 'Data Envio',
    'Modo Envio', 'Prioridade Pedido',
    
    # Produto
    'ID Produto', 'Categoria', 'SubCategoria', 'Nome Produto',
    
    # Vendas
    'Valor Venda', 'Quantidade Vendida', 'Custo Envio'
]

super_dataset = super_dataset[colunas_organizadas]

In [18]:
#super_dataset.to_csv('1-datasets/dataset_completo.csv', index=False, encoding='utf-8')

## Perguntas de Negocio

### 1 - Qual foi o total de valor venda considerando cada modo de envio dos pedidos?

Colunas Necessárias:
- Valor Vendas
- Modo Envio

In [19]:
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [20]:
valor_total_de_vendas_por_modo_de_envio = super_dataset.groupby(['Modo Envio'])['Valor Venda'].sum().reset_index().sort_values('Valor Venda', ascending=False)
valor_total_de_vendas_por_modo_de_envio

,Modo Envio,Valor Venda
0,Classe Padrao,"7,556,488.21"
3,Segunda Classe,"2,565,561.05"
2,Primeira Classe,"1,843,147.32"
1,Mesmo Dia,"677,305.33"


### 2 - Quais mercados tiveram o maior custo médio de envio dos produtos vendidos?

Colunas Necessárias:
- Mercado
- Custo Envio

In [21]:
custo_medio_de_envios_por_mercado = super_dataset.groupby(['Mercado'])['Custo Envio'].mean().reset_index().sort_values('Custo Envio', ascending=False)
custo_medio_de_envios_por_mercado

,Mercado,Custo Envio
0,APAC,29.14
6,US,28.94
4,EU,27.84
5,LATAM,26.46
1,Africa,19.44
2,Canada,17.78
3,EMEA,17.48


### 3-A empresa tem como objetivo (meta) manter uma média de 350 para o valor de venda todos os meses.
 
 - Mostre um indicador (KPI–Key Performance Indicator) com o valor médio de venda.

 - A empresa ficou abaixo ou acima da meta no mês de Abril/2014?

Colunas Necessárias:
- Data Pedido
- Valor Venda

In [22]:
valor_medio_de_venda_por_mes = (super_dataset.groupby(super_dataset['Data Pedido'].dt.to_period('M'))['Valor Venda'].mean().reset_index())
valor_medio_de_venda_por_mes

,Data Pedido,Valor Venda
0,2011-01,227.58
1,2011-02,249.60
2,2011-03,272.26
3,2011-04,206.85
4,2011-05,257.46
5,2011-06,235.42
6,2011-07,232.53
7,2011-08,237.13
8,2011-09,274.67
9,2011-10,258.16


In [23]:
valor_medio_de_venda_por_mes[valor_medio_de_venda_por_mes['Data Pedido'] == '2014-04']

,Data Pedido,Valor Venda
39,2014-04,229.62


### 4 - Qual categoria de produto apresentou maior lucro médio?

- Considere que o lucro é equivalente a: valor venda - custo envio.

Colunas Necessárias:
- Valor Venda
- Custo Envio
- Categoria

In [24]:
super_dataset['Lucro'] = super_dataset['Valor Venda'] - super_dataset['Custo Envio']
lucro_medio_por_categoria = super_dataset.groupby(['Categoria'])['Lucro'].mean().reset_index().sort_values('Lucro', ascending=False)
lucro_medio_por_categoria

,Categoria,Lucro
2,Tecnologia,417.86
1,Moveis,371.66
0,Material de Escritorio,108.13


### 5 - Qual foi o comportamento da margem de lucro ao longo do tempo?

 - Considere a margem de lucro como o lucro dividido pelo valor venda.

Colunas Necessárias:
- Data Pedido
- Valor Venda
- Custo Envio

In [25]:
super_dataset['Margem Lucro'] = super_dataset['Lucro'] / super_dataset['Valor Venda']
valor_margem_de_lucro_pelo_tempo = super_dataset.groupby(super_dataset['Data Pedido'].dt.to_period('M'))['Margem Lucro'].sum().reset_index()
valor_margem_de_lucro_pelo_tempo

,Data Pedido,Margem Lucro
0,2011-01,389.46
1,2011-02,320.41
2,2011-03,474.60
3,2011-04,508.94
4,2011-05,497.92
5,2011-06,814.15
6,2011-07,432.11
7,2011-08,776.76
8,2011-09,942.22
9,2011-10,691.44


#### Grafico para exemplificar

In [26]:
df = valor_margem_de_lucro_pelo_tempo.copy()
df['Data Pedido'] = df['Data Pedido'].astype(str)
df = df.sort_values('Data Pedido').reset_index(drop=True)

In [27]:
x_num = np.arange(len(df))
y = df['Margem Lucro'].values

In [28]:
# Regressão linear
x_const = sm.add_constant(x_num)
modelo = sm.OLS(y, x_const).fit()
linha_regressao = modelo.predict(x_const)

In [29]:
# Desvio padrão e bandas
std = np.std(y)
banda_superior = linha_regressao + 1 * std
banda_inferior = linha_regressao - 1 * std

In [30]:
# Tendência
slope = modelo.params[1]
tendencia = 'Crescendo ↑' if slope > 0 else 'Decrescendo ↓' if slope < 0 else 'Estável →'


In [31]:
grafico_margem_de_lucro_vs_tempo = valor_margem_de_lucro_pelo_tempo.copy()
grafico_margem_de_lucro_vs_tempo['Data Pedido'] = grafico_margem_de_lucro_vs_tempo['Data Pedido'].astype(str)
grafico_margem_de_lucro_vs_tempo = grafico_margem_de_lucro_vs_tempo.sort_values('Data Pedido').reset_index(drop=True)
grafico_comprimento_eixo_x = np.arange(len(grafico_margem_de_lucro_vs_tempo))
dimenção_y = grafico_margem_de_lucro_vs_tempo['Margem Lucro'].values

# Regressão linear
constante_eixo_x = sm.add_constant(grafico_comprimento_eixo_x)
modelo = sm.OLS(dimenção_y, constante_eixo_x).fit()
linha_regressao = modelo.predict(constante_eixo_x)

# Desvio padrão e bandas
desvio_padrao = np.std(dimenção_y)
banda_superior = linha_regressao + 1 * desvio_padrao
banda_inferior = linha_regressao - 1 * desvio_padrao

# Tendência
inclinacao = modelo.params[1]
tendencia = 'Crescendo ↑' if inclinacao > 0 else 'Decrescendo ↓' if inclinacao < 0 else 'Estável →'

# Antes de criar o go.Figure()
datas_completas = grafico_margem_de_lucro_vs_tempo['Data Pedido'].tolist()

eixo_x_labels = grafico_margem_de_lucro_vs_tempo['Data Pedido'].tolist()

# Mostra só o ano em janeiro, vazio nos outros meses
eixo_x_ticks = [label[:4] if label.endswith('-01') else '' for label in eixo_x_labels]
figura = go.Figure()

# Banda Superior — esconde no hover
figura.add_trace(go.Scatter(
    x=eixo_x_labels, y=banda_superior,
    line=dict(width=0), showlegend=False, name='Banda Superior',
    hoverinfo='skip'
))

# Banda Inferior — esconde no hover
figura.add_trace(go.Scatter(
    x=eixo_x_labels, y=banda_inferior,
    fill='tonexty', fillcolor='rgba(53,161,165,0.15)',
    line=dict(width=0), name='±1 Desvio Padrão',
    hoverinfo='skip'
))

# Margem de Lucro — mostra mês/ano e valor
figura.add_trace(go.Scatter(
    x=eixo_x_labels, y=dimenção_y,
    mode='lines+markers',
    name='Margem de Lucro',
    line=dict(color='#35a1a5', width=2),
    marker=dict(size=5),
    customdata=datas_completas,
    hovertemplate='%{customdata}<br>Margem: %{y:.2f}<extra></extra>'
))

# Regressão — mostra só a tendência
figura.add_trace(go.Scatter(
    x=eixo_x_labels, y=linha_regressao,
    mode='lines',
    name=f'Tendência: {tendencia}',
    line=dict(color='#2196f3', width=2, dash='dash'),
    hoverinfo='skip'
))

figura.update_layout(
    title='Margem de Lucro ao Longo do Tempo',
    xaxis=dict(
        tickangle=0,
        tickvals=eixo_x_labels,
        ticktext=eixo_x_ticks,
        showgrid=False,
        tickfont=dict(color='black')
    ),
    yaxis=dict(
        showgrid=True,
        tickfont=dict(color='black')
    ),
    hovermode='x unified',
    showlegend=False,
    plot_bgcolor='rgba(33, 150, 243, 0)',   # #2196f3 com 40% transparência
    paper_bgcolor='rgba(33, 150, 243, 0)',
    font=dict(color='black'),         # todas as letras pretas
    hoverlabel=dict(
        bgcolor='white',
        font=dict(color='black')
    ),
    modebar_remove=['zoom', 'pan', 'select', 'lasso2d', 'zoomIn2d', 'zoomOut2d', 'autoScale2d', 'resetScale2d', 'toImage']
)



figura.show()

# Exportar HTML para o GitHub
figura.write_html(
    'margem_lucro.html',
    include_plotlyjs='cdn',
    config={'responsive': True},
    full_html=True,
    post_script="""
        document.body.style.backgroundColor = 'rgba(33, 150, 243, 0.25)';
        document.body.style.margin = '0';
        document.body.style.padding = '0';
    """
)